# ChEBI — Chemical Entities of Biological Interest

**ChEBI** (Chemical Entities of Biological Interest) is a freely available dictionary and ontology of molecular entities focused on small chemical compounds that are relevant to biology. It is maintained by the European Bioinformatics Institute (EMBL-EBI) and is part of the ELIXIR Core Data Resources.

Each entry provides:

| Field | Description |
|---|---|
| `chebi_id` | Stable identifier, e.g. `CHEBI:15422` |
| `name` | IUPAC or recommended common name |
| `formula` | Molecular formula |
| `monoisotopic_mass` | Mass calculated from most-abundant isotopes (Da) |
| `charge` | Net formal charge |
| `inchi` | IUPAC International Chemical Identifier (unique structure string) |
| `smiles` | Simplified Molecular Input Line Entry System string |
| `star_rating` | Annotation quality: 3★ = manually annotated, 1★ = automatic |

ChEBI also ships an **OBO-format ontology** encoding `is_a`, `has_role`, `has_part`, and other relationships, enabling pathway and role-based queries.

**API base URL:** `https://www.ebi.ac.uk/webservices/chebi/2.0/test/`  
**OLS4 search:** `https://www.ebi.ac.uk/ols4/api/search?q={term}&ontology=chebi`  
**OBO bulk download:** `https://ftp.ebi.ac.uk/pub/databases/chebi/ontology/chebi.obo.gz`

**Reference:** Hastings et al. (2016), *Nucleic Acids Research*, ChEBI in 2016: Improved services and an enhanced role in the ELIXIR infrastructure.

# TODO

* [x] **Ingest data**
    * [x] Connect to ChEBI REST API and fetch a single well-known compound (ATP, CHEBI:15422)
    * [x] Fetch metadata for a curated list of biologically important metabolites
    * [x] Cache compound data to `data/chebi_compounds.json`
    * [x] Parse into a Polars DataFrame with typed columns
    * [x] Download and parse a section of the ChEBI OBO ontology (lipid subtree)
    * [x] Build a parent–child ontology relationship DataFrame
* [ ] **Explore and clean**
    * [ ] Summarise dataset dimensions, missing values, and star-rating distribution
    * [ ] Examine mass and charge distributions across compound classes
    * [ ] Inspect and prune the lipid ontology subgraph
* [ ] **Analysis**
    * [ ] Classify compounds by role (metabolite, cofactor, neurotransmitter, lipid, etc.)
    * [ ] Traverse ontology hierarchy to enumerate all lipid sub-classes
    * [ ] Compare monoisotopic masses to average masses and discuss isotope effects
* [ ] **Visualization**
    * [ ] Plot mass distribution by compound class
    * [ ] Render the lipid ontology subgraph as a directed network
    * [ ] Plot charge vs. mass coloured by biological role
* [ ] **Statistical analysis**
    * [ ] Test whether annotation quality (star rating) correlates with structural complexity
    * [ ] Discuss isotope mass accuracy and error propagation for formula-derived masses
    * [ ] Multiple-comparison considerations when querying large ontology subtrees

In [ ]:
import json
import time
import gzip
import io
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import requests
import polars as pl

## 1. Ingest Data

### 1.1 Connect to ChEBI API — Fetch a Single Compound

In [ ]:
CHEBI_REST = "https://www.ebi.ac.uk/webservices/chebi/2.0/test"
# XML namespace used in all ChEBI SOAP/REST responses
NS = {"chebi": "https://www.ebi.ac.uk/webservices/chebi"}


def get_complete_entity(chebi_id: str) -> dict:
    """Fetch a complete ChEBI entity record via the REST endpoint.

    Parameters
    ----------
    chebi_id : str
        ChEBI identifier, e.g. ``"CHEBI:15422"``.

    Returns
    -------
    dict
        Parsed entity fields: chebi_id, name, formula, monoisotopic_mass,
        charge, inchi, smiles, star_rating.

    Raises
    ------
    requests.HTTPError
        If the HTTP request fails.
    """
    url = f"{CHEBI_REST}/getCompleteEntity"
    resp = requests.get(url, params={"chebiId": chebi_id}, timeout=30)
    resp.raise_for_status()

    # ChEBI returns a SOAP envelope — parse the XML tree
    root = ET.fromstring(resp.text)

    def txt(tag: str) -> str:
        """Extract text from the first matching tag in the chebi namespace."""
        node = root.find(f".//chebi:{tag}", NS)
        return node.text.strip() if node is not None and node.text else ""

    # Monoisotopic mass is nested inside <ChemicalDataItem>
    mono_mass = ""
    for item in root.findall(".//chebi:ChemicalDataItem", NS):
        dtype = item.find("chebi:datatype", NS)
        if dtype is not None and dtype.text == "MONOISOTOPIC MASS":
            val = item.find("chebi:data", NS)
            mono_mass = val.text.strip() if val is not None else ""
            break

    return {
        "chebi_id": txt("chebiId"),          # stable ChEBI accession
        "name": txt("chebiAsciiName"),        # recommended name
        "formula": txt("Formulae"),           # molecular formula string
        "monoisotopic_mass": mono_mass,       # exact mass from isotopes
        "charge": txt("charge"),              # net formal charge
        "inchi": txt("inchi"),                # canonical structure identifier
        "smiles": txt("smiles"),              # SMILES structure string
        "star_rating": txt("entityStar"),     # annotation quality (1–3)
    }


# ── Connectivity check: fetch ATP ────────────────────────────────────────────
atp = get_complete_entity("CHEBI:15422")
print("=== ATP (CHEBI:15422) ===")
for k, v in atp.items():
    # Truncate long InChI/SMILES for readability
    display_v = v[:80] + "..." if len(v) > 80 else v
    print(f"  {k:<20}: {display_v}")

### 1.2 Fetch a Curated List of Biologically Important Metabolites

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
COMPOUNDS_PATH = DATA_DIR / "chebi_compounds.json"

# Curated list: ChEBI ID → human-readable label (for progress messages)
TARGET_COMPOUNDS = {
    "CHEBI:15422": "ATP",
    "CHEBI:16908": "NADH",
    "CHEBI:17234": "glucose",
    "CHEBI:18012": "dopamine",
    "CHEBI:16113": "cholesterol",
    "CHEBI:15361": "pyruvate",
    "CHEBI:16761": "ADP",
    "CHEBI:17659": "GTP",
    "CHEBI:15379": "dioxygen",
    "CHEBI:16908": "NADH",
    "CHEBI:16474": "NAD+",
    "CHEBI:30616": "ATP (alternate)",
    "CHEBI:17895": "tyrosine",
    "CHEBI:17822": "serine",
    "CHEBI:16015": "acetyl-CoA",
    "CHEBI:24996": "lactate",
    "CHEBI:18420": "magnesium ion",
    "CHEBI:29101": "sodium ion",
    "CHEBI:29103": "potassium ion",
    "CHEBI:26020": "phosphate",
}


def fetch_compound_list(chebi_ids: dict, cache_path: Path) -> list[dict]:
    """Fetch ChEBI records for a collection of compound IDs with disk caching.

    If ``cache_path`` already exists, the cached data is returned immediately
    without any network requests.

    Parameters
    ----------
    chebi_ids : dict
        Mapping of ChEBI ID strings to descriptive labels (used in progress
        messages only).
    cache_path : Path
        Filesystem path for the JSON cache file.

    Returns
    -------
    list[dict]
        One dict per compound with keys: chebi_id, name, formula,
        monoisotopic_mass, charge, inchi, smiles, star_rating.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        with open(cache_path) as f:
            return json.load(f)

    records = []
    unique_ids = list(dict.fromkeys(chebi_ids))  # preserve order, deduplicate
    for chebi_id in unique_ids:
        label = chebi_ids[chebi_id]
        print(f"  Fetching {chebi_id} ({label}) ...", end=" ")
        try:
            rec = get_complete_entity(chebi_id)
            records.append(rec)
            print("ok")
        except Exception as exc:
            print(f"ERROR: {exc}")
        time.sleep(0.3)  # be polite to the EBI servers

    # Persist to disk so subsequent runs skip the network round-trips
    with open(cache_path, "w") as f:
        json.dump(records, f, indent=2)
    print(f"Cached {len(records)} records to {cache_path}")
    return records


records = fetch_compound_list(TARGET_COMPOUNDS, COMPOUNDS_PATH)
print(f"\nTotal records fetched: {len(records)}")

### 1.3 Parse into a Polars DataFrame

In [ ]:
# Build DataFrame directly from the list of dicts
compounds_df = pl.DataFrame(records)

# Cast numeric columns from string to appropriate types
compounds_df = compounds_df.with_columns([
    # monoisotopic_mass: float; empty string → null
    pl.col("monoisotopic_mass")
      .replace("", None)
      .cast(pl.Float64),
    # charge: integer; empty string → null
    pl.col("charge")
      .replace("", None)
      .cast(pl.Int32),
    # star_rating: integer annotation quality (1–3)
    pl.col("star_rating")
      .replace("", None)
      .cast(pl.Int32),
])

print(f"Shape : {compounds_df.shape}")
print(f"\nDtypes:")
for col, dtype in zip(compounds_df.columns, compounds_df.dtypes):
    print(f"  {col:<22} {dtype}")
print()
compounds_df.head(5)

### 1.4 Download and Parse the ChEBI OBO Ontology (Lipid Subtree)

In [ ]:
OBO_URL = "https://ftp.ebi.ac.uk/pub/databases/chebi/ontology/chebi.obo.gz"
OBO_PATH = DATA_DIR / "chebi.obo.gz"

if not OBO_PATH.exists():
    print(f"Downloading {OBO_URL} ...")
    with requests.get(OBO_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(OBO_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="\r")
    print(f"\nSaved to {OBO_PATH}")
else:
    print(f"Already downloaded: {OBO_PATH}")

### 1.5 Parse OBO into Parent–Child Relationship DataFrame

In [ ]:
def parse_obo_terms(obo_path: Path) -> pl.DataFrame:
    """Parse a gzip-compressed OBO file into a DataFrame of ontology terms.

    Only ``[Term]`` stanzas are processed.  Relationship types extracted:
    ``is_a``, ``relationship: has_role``, ``relationship: has_part``.

    Parameters
    ----------
    obo_path : Path
        Path to the ``.obo.gz`` file.

    Returns
    -------
    pl.DataFrame
        Columns: child_id (str), child_name (str), parent_id (str),
        relation (str).  One row per (child, parent) edge.
    """
    rows = []          # accumulate (child_id, child_name, parent_id, relation)
    current: dict = {} # buffer for the current [Term] block

    with gzip.open(obo_path, "rt", encoding="utf-8") as fh:
        for raw_line in fh:
            line = raw_line.rstrip()

            if line == "[Term]":
                # Flush the previous term's edges before starting a new one
                if current.get("id"):
                    for parent_id, rel in current.get("parents", []):
                        rows.append({
                            "child_id": current["id"],
                            "child_name": current.get("name", ""),
                            "parent_id": parent_id,
                            "relation": rel,
                        })
                current = {"parents": []}

            elif line.startswith("id: "):
                current["id"] = line[4:]           # e.g. "CHEBI:18059"

            elif line.startswith("name: "):
                current["name"] = line[6:]          # human-readable label

            elif line.startswith("is_a: "):
                # Format: "is_a: CHEBI:XXXXX ! parent name"
                parent_id = line[6:].split(" ")[0].strip()
                current["parents"].append((parent_id, "is_a"))

            elif line.startswith("relationship: "):
                # Format: "relationship: has_role CHEBI:XXXXX ! label"
                parts = line[14:].split(" ")
                if len(parts) >= 2:
                    rel_type = parts[0]             # e.g. "has_role"
                    parent_id = parts[1].strip()    # e.g. "CHEBI:25212"
                    current["parents"].append((parent_id, rel_type))

    # Flush the very last term in the file
    if current.get("id"):
        for parent_id, rel in current.get("parents", []):
            rows.append({
                "child_id": current["id"],
                "child_name": current.get("name", ""),
                "parent_id": parent_id,
                "relation": rel,
            })

    return pl.DataFrame(rows, schema={
        "child_id": pl.Utf8,
        "child_name": pl.Utf8,
        "parent_id": pl.Utf8,
        "relation": pl.Utf8,
    })


print("Parsing OBO file (this may take ~30 s) ...")
obo_df = parse_obo_terms(OBO_PATH)
print(f"Total ontology edges parsed : {len(obo_df):,}")
print(f"Unique child terms          : {obo_df['child_id'].n_unique():,}")
print(f"Relation types              : {obo_df['relation'].unique().to_list()}")
print()
obo_df.head(5)

### 1.6 Extract the Lipid Ontology Subtree (CHEBI:18059)

In [ ]:
LIPID_ROOT = "CHEBI:18059"  # "lipid" in ChEBI


def descendants_bfs(obo_edges: pl.DataFrame, root_id: str,
                    relation: str = "is_a") -> set[str]:
    """Collect all descendants of ``root_id`` via BFS over ``is_a`` edges.

    The OBO edges are stored as child→parent, so we invert the traversal:
    we look for rows where ``parent_id == current`` and collect ``child_id``.

    Parameters
    ----------
    obo_edges : pl.DataFrame
        Full ontology edge table (child_id, child_name, parent_id, relation).
    root_id : str
        ChEBI ID of the subtree root, e.g. ``"CHEBI:18059"``.
    relation : str, optional
        Relation type to follow (default ``"is_a"``).

    Returns
    -------
    set[str]
        Set of all descendant ChEBI IDs (excluding the root itself).
    """
    # Pre-filter to the requested relation type for speed
    is_a_edges = obo_edges.filter(pl.col("relation") == relation)

    # Build a parent→[children] lookup as a plain Python dict
    parent_to_children: dict[str, list[str]] = {}
    for row in is_a_edges.select(["child_id", "parent_id"]).iter_rows():
        child, parent = row
        parent_to_children.setdefault(parent, []).append(child)

    visited: set[str] = set()
    queue = [root_id]
    while queue:
        node = queue.pop()
        for child in parent_to_children.get(node, []):
            if child not in visited:
                visited.add(child)
                queue.append(child)
    return visited


# Collect all lipid descendants
lipid_ids = descendants_bfs(obo_df, LIPID_ROOT)
print(f"Lipid subtree size: {len(lipid_ids):,} terms beneath {LIPID_ROOT}")

# Filter the edge table to keep only edges within the lipid subtree
lipid_ids_with_root = lipid_ids | {LIPID_ROOT}
lipid_edges = obo_df.filter(
    pl.col("child_id").is_in(lipid_ids_with_root) &
    pl.col("parent_id").is_in(lipid_ids_with_root) &
    (pl.col("relation") == "is_a")
)

print(f"Edges within lipid subtree : {len(lipid_edges):,}")
print(f"\nShape : {lipid_edges.shape}")
print(f"\nDtypes:")
for col, dtype in zip(lipid_edges.columns, lipid_edges.dtypes):
    print(f"  {col:<12} {dtype}")
print()
lipid_edges.head(10)